# ROGII Wellbore Geology Prediction — Physics-Informed ML Champion Ensemble (Kriging version)

This notebook implements our physics-informed, un-leaked geologic dipping plane model combined with a high-performance **weighted ensemble (LightGBM + CatBoost + Random Forest)** to correct local micro-stratigraphic residuals.

### Key Technical Breakthroughs:
1. **Raw API Mismatch Corridor:** Bypasses local GR percentile standardization (which degrades performance due to measurement noise distortion in homogeneous buildup zones) to retain a raw API mismatch corridor.
2. **3D Geological Surface Mapping via Kriging:** Precomputes stable regional neighbor 3D point coordinates and fits a localized Gaussian Process (Kriging) model to map the undulating boundary surface, anchored exactly at the last known true TVT point to guarantee 100% geologic continuity.
3. **LightGBM + CatBoost + Random Forest Ensemble:** Predicts micro-structural residuals with a weighted blend (50% LightGBM + 30% CatBoost + 20% Random Forest).
4. **Dynamic Training Speedups:** Automatically uses T4/P100 **GPU acceleration** for CatBoost (with try-except CPU fallback) and dynamically enables full training on all **773 wells** on Kaggle, while keeping local validation blindingly fast.

## Imports & Setup


In [ ]:
"""
ROGII v5 — Wellbore Geology Prediction
======================================
Physics-Informed ML: 3D Spatial Plane Baseline (Neighbor Regularized) + Local Type Log Stratigraphic Features + LightGBM Residual Corrector
Directly optimized for RMSE. Blindingly fast and mathematically robust.
"""
import os, time, gc, warnings
import numpy as np
import pandas as pd
from scipy import interpolate, signal
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings("ignore")
np.random.seed(42)


## Config


In [ ]:
class CFG:
    SEED = 42
    N_FOLDS = 5
    NUM_BOOST = 1500
    EARLY_STOP = 50
    SMOOTH_WIN = 31
    SMOOTH_POLY = 3
    ALPHA_BLEND = 0.1  # Optimal blend: 90% Neighbor Slopes + 10% Local Slopes
    FAST_LOCAL = not os.path.exists('/kaggle/input')  # Subset training wells locally to verify pipeline instantly
    
    # Paths
    TRAIN_DIR = TEST_DIR = SAMPLE_SUB = OUTPUT_DIR = None
    if os.path.exists('/kaggle/input'):
        OUTPUT_DIR = '/kaggle/working'
        for root, dirs, files in os.walk('/kaggle/input'):
            if any('__horizontal_well.csv' in f for f in files):
                bn = os.path.basename(root)
                if bn == 'train': TRAIN_DIR = root
                elif bn == 'test': TEST_DIR = root
            if 'sample_submission.csv' in files:
                SAMPLE_SUB = os.path.join(root, 'sample_submission.csv')
        if TRAIN_DIR and not TEST_DIR:
            TEST_DIR = os.path.join(os.path.dirname(TRAIN_DIR), 'test')
        if not SAMPLE_SUB and TRAIN_DIR:
            SAMPLE_SUB = os.path.join(os.path.dirname(TRAIN_DIR), 'sample_submission.csv')
        if not TRAIN_DIR:
            TRAIN_DIR = '/kaggle/input/train'; TEST_DIR = '/kaggle/input/test'
            SAMPLE_SUB = '/kaggle/input/sample_submission.csv'
    else:
        _d = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
        TRAIN_DIR = os.path.join(_d, 'data', 'train')
        TEST_DIR = os.path.join(_d, 'data', 'test')
        SAMPLE_SUB = os.path.join(_d, 'data', 'sample_submission.csv')
        OUTPUT_DIR = _d

    LGB_PARAMS = dict(
        objective='regression_l2',  # Minimize RMSE directly
        metric='rmse',
        boosting_type='gbdt',
        learning_rate=0.01,         # Slow learning rate to prevent overfitting
        max_depth=4,                # Shallow trees for stable generalization
        num_leaves=15,
        min_child_samples=100,      # Large leaf size to suppress high-frequency noise
        reg_alpha=1.0,              # L1 regularization
        reg_lambda=10.0,            # L2 regularization
        verbose=-1,
        seed=SEED,
        n_jobs=-1,
    )


## Core Utilities


In [ ]:
def get_well_names(d):
    return sorted(set(f.split('__')[0] for f in os.listdir(d) if '__horizontal_well.csv' in f))

def load_well(wid, d):
    hw = pd.read_csv(os.path.join(d, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(d, f'{wid}__typewell.csv'))
    return hw, tw

def fill_gr(gr):
    return gr.interpolate('linear', limit_direction='both').ffill().bfill().fillna(0.0)


## Spatial Neighbor Dip & Slope Precomputation (PPT Slide 12-13)


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

def precompute_spatial_slopes():
    """
    Extracts representative 3D points (X, Y, Z_form) from all training wells
    to fit local Kriging (Gaussian Process) surfaces for target wells.
    Redefined to use Strategy 3's Kriging mapping while keeping the signature identical.
    """
    wids = get_well_names(CFG.TRAIN_DIR)
    well_points = []
    well_centroids = []
    
    print("[KRIGING] Extracting 3D geological boundary points from training wells...")
    for wid in wids:
        try:
            hw, _ = load_well(wid, CFG.TRAIN_DIR)
            if 'TVT' not in hw.columns or len(hw) < 50:
                continue
            
            known = hw[hw['TVT'].notna()]
            if len(known) < 10:
                continue
                
            # Downsample to 5 representative points along the wellbore to keep it extremely fast
            indices = np.linspace(0, len(known) - 1, 5, dtype=int)
            pts_x = known['X'].values[indices]
            pts_y = known['Y'].values[indices]
            pts_z_form = (known['Z'].values + known['TVT'].values)[indices]
            
            for x, y, z_f in zip(pts_x, pts_y, pts_z_form):
                well_points.append([x, y, z_f, wid])
                
            well_centroids.append({
                'well_id': wid,
                'cx': hw['X'].mean(),
                'cy': hw['Y'].mean()
            })
        except:
            continue
            
    df_pts = pd.DataFrame(well_points, columns=['X', 'Y', 'Z_form', 'well_id'])
    df_centroids = pd.DataFrame(well_centroids)
    centroids = df_centroids[['cx', 'cy']].values
    tree = cKDTree(centroids)
    
    # Store both centroids and points in a dictionary
    df_slopes_dict = {
        'centroids': df_centroids,
        'points': df_pts
    }
    
    return tree, df_slopes_dict


def estimate_well_plane(tree, df_slopes_dict, hw, tvt_input_col='TVT_input'):
    """
    Predicts the stable 3D boundary surface using a local Kriging (Gaussian Process) fit
    on the points of the 6 nearest neighbor wells.
    Redefined to use Strategy 3's Kriging mapping while keeping the signature identical.
    """
    cx, cy = hw['X'].mean(), hw['Y'].mean()
    known = hw[hw[tvt_input_col].notna()]
    
    df_centroids = df_slopes_dict['centroids']
    df_pts = df_slopes_dict['points']
    
    # Query nearest neighbor wells
    dist, idxs = tree.query([cx, cy], k=7)
    neighbor_wids = []
    for d, i in zip(dist, idxs):
        wid = df_centroids.loc[i, 'well_id']
        # Exclude self to prevent target well data leakage from training labels
        if d < 1e-3:
            continue
        neighbor_wids.append(wid)
        
    neighbor_wids = neighbor_wids[:6]
    
    # Extract training points from these neighbor wells
    nb_pts = df_pts[df_pts['well_id'].isin(neighbor_wids)]
    X_train = nb_pts[['X', 'Y']].values
    y_train = nb_pts['Z_form'].values
    
    # Fit local Gaussian Process (Kriging)
    gp = GaussianProcessRegressor(
        kernel=Matern(length_scale=1000.0, length_scale_bounds=(200.0, 5000.0), nu=1.5),
        alpha=0.1,
        normalize_y=True,
        random_state=42
    )
    gp.fit(X_train, y_train)
    
    # Predict the boundary surface Z_form along the entire well path (X, Y)
    X_pred = hw[['X', 'Y']].values
    z_form_pred = gp.predict(X_pred)
    
    # Anchor the intercept exactly at the last known true TVT point to guarantee physical continuity
    last_row = known.iloc[-1]
    last_idx = len(known) - 1
    C_shift = (last_row['Z'] + last_row[tvt_input_col]) - z_form_pred[last_idx]
    
    # Apply Shift
    z_form_pred = z_form_pred + C_shift
    tvt_plane = z_form_pred - hw['Z'].values
    
    # Compute proxy linear gradients for compatibility
    slope_x = (z_form_pred[-1] - z_form_pred[0]) / (hw['X'].iloc[-1] - hw['X'].iloc[0] + 1e-5)
    slope_y = (z_form_pred[-1] - z_form_pred[0]) / (hw['Y'].iloc[-1] - hw['Y'].iloc[0] + 1e-5)
    
    return tvt_plane, slope_x, slope_y


## Local Type Log Construction (PPT Slide 9)


In [ ]:
def build_local_typelog(hw, tw, ps_idx):
    known = hw.iloc[:ps_idx+1].dropna(subset=['TVT_input', 'GR'])
    known = known.sort_values('TVT_input')
    
    tw_clean = tw.dropna(subset=['TVT', 'GR']).sort_values('TVT')
    if len(tw_clean) < 2: return None, None
    tw_interp = interpolate.interp1d(tw_clean['TVT'], tw_clean['GR'], bounds_error=False, fill_value=(tw_clean['GR'].iloc[0], tw_clean['GR'].iloc[-1]))
    
    min_tvt = min(known['TVT_input'].min(), tw_clean['TVT'].min()) if len(known)>0 else tw_clean['TVT'].min()
    max_tvt = max(known['TVT_input'].max(), tw_clean['TVT'].max()) if len(known)>0 else tw_clean['TVT'].max()
    min_tvt -= 200.0; max_tvt += 400.0  # Safe TVT range buffer
    
    tvt_axis = np.arange(min_tvt, max_tvt, 0.5)
    typelog_gr = tw_interp(tvt_axis)
    
    if len(known) > 10:
        known_agg = known.groupby('TVT_input')['GR'].mean().reset_index().sort_values('TVT_input')
        local_interp = interpolate.interp1d(known_agg['TVT_input'], known_agg['GR'], bounds_error=False)
        local_vals = local_interp(tvt_axis)
        valid = ~np.isnan(local_vals)
        # 80% Local Type Log blend (higher resolution) + 20% fallback Typewell
        typelog_gr[valid] = 0.8 * local_vals[valid] + 0.2 * typelog_gr[valid]
        
    typelog_gr = signal.savgol_filter(typelog_gr, 11, 2)
    # DO NOT standardize typelog_gr (keep raw API scale for physical mismatch mapping)
    return tvt_axis, typelog_gr


## Feature Engineering & Pipeline


In [ ]:
def find_ps_train(hw):
    if 'TVT_input' not in hw.columns: return len(hw)//3
    mask = hw['TVT_input'].notna()
    return int(mask.values.nonzero()[0][-1]) if mask.any() else len(hw)//3

def find_ps_test(sub_df, wid):
    rows = sub_df[sub_df['well_id'] == wid]
    return int(rows['row_idx'].min()) - 1 if len(rows) > 0 else None

def build_features(hw, tw, ps_idx, tree, df_slopes, is_train=True):
    n = len(hw)
    md = hw['MD'].values
    gr = fill_gr(hw['GR']).values
    
    known_tvt = hw['TVT'].values[:ps_idx+1] if 'TVT' in hw.columns else hw['TVT_input'].iloc[:ps_idx+1].values
    
    # 1. Physics baseline dipping plane prediction (neighbor regularized, unleaked)
    tvt_plane, slope_x, slope_y = estimate_well_plane(tree, df_slopes, hw)
    
    tvt_baseline = np.zeros(n)
    tvt_baseline[:ps_idx+1] = known_tvt
    tvt_baseline[ps_idx+1:] = tvt_plane[ps_idx+1:]
    
    # 2. Local Type Log construction
    tvt_axis, typelog_gr = build_local_typelog(hw, tw, ps_idx)
    
    # Build Feature DataFrame
    df = pd.DataFrame(index=range(n))
    df['dMD'] = np.diff(md, prepend=md[0]).clip(0.1)
    df['dX'] = hw['X'].diff().fillna(0).values
    df['dY'] = hw['Y'].diff().fillna(0).values
    df['dZ'] = hw['Z'].diff().fillna(0).values
    df['incl'] = df['dZ'] / df['dMD']
    df['dogleg'] = df['incl'].diff().fillna(0)
    
    # GR signals
    df['gr'] = gr
    df['gr_diff'] = np.diff(gr, prepend=gr[0])
    s = pd.Series(gr)
    for w in [15, 31, 61]:
        df[f'gr_roll_{w}'] = s.rolling(w, center=True, min_periods=1).mean().values
        df[f'gr_std_{w}'] = s.rolling(w, center=True, min_periods=1).std().fillna(0).values
        
    df['tvt_baseline'] = tvt_baseline
    df['tvt_plane'] = tvt_plane
    df['dist_from_ps'] = (md - md[ps_idx]).clip(0)
    df['is_pred_zone'] = (np.arange(n) > ps_idx).astype(np.int8)
    
    # 3. Stratigraphic Local Type Log GR features (Virtual pattern matching corridor)
    if tvt_axis is not None:
        typelog_interp = interpolate.interp1d(tvt_axis, typelog_gr, bounds_error=False, fill_value=(typelog_gr[0], typelog_gr[-1]))
        df['tw_gr_at_plane'] = typelog_interp(tvt_plane)
        df['tw_gr_diff'] = gr - df['tw_gr_at_plane']
        
        # Dense offset grid around baseline to capture local stratigraphy profile
        for offset in [-50, -40, -30, -20, -10, -5, -2, 2, 5, 10, 20, 30, 40, 50]:
            df[f'tw_gr_offset_{offset}'] = typelog_interp(tvt_plane + offset)
            df[f'tw_gr_diff_offset_{offset}'] = gr - df[f'tw_gr_offset_{offset}']
    else:
        df['tw_gr_at_plane'] = 0.0
        df['tw_gr_diff'] = 0.0
        for offset in [-50, -40, -30, -20, -10, -5, -2, 2, 5, 10, 20, 30, 40, 50]:
            df[f'tw_gr_offset_{offset}'] = 0.0
            df[f'tw_gr_diff_offset_{offset}'] = 0.0
            
    if is_train and 'TVT' in hw.columns:
        df['target'] = hw['TVT'].values - tvt_baseline
        df['tvt_true'] = hw['TVT'].values
        
    return df


## Feature Extraction Helpers


In [ ]:
FEAT_COLS = None

def get_feat_cols(df):
    # Exclude coordinate targets and coordinates to prevent regional overfitting
    return [c for c in df.columns if c not in {'TVT', 'TVT_input', 'target', 'tvt_true', 'group', 'is_pred_zone', 'X', 'Y', 'Z', 'MD', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'GR', 'tvt_baseline', 'tvt_plane'}]

def prepare_train(tree, df_slopes):
    wids = get_well_names(CFG.TRAIN_DIR)
    if CFG.FAST_LOCAL:
        wids = wids[:80]
    print(f"[DATA] Loading and precomputing {len(wids)} training wells...")
    dfs = []
    
    for i, wid in enumerate(wids):
        hw, tw = load_well(wid, CFG.TRAIN_DIR)
        ps = find_ps_train(hw)
        feat = build_features(hw, tw, ps, tree, df_slopes, is_train=True)
        sub = feat.iloc[ps+1:].copy()  # Only keep prediction zone rows for training LightGBM
        sub['group'] = i
        dfs.append(sub)
        
    combined = pd.concat(dfs, ignore_index=True)
    return combined


## GroupKFold Model Training Loop


In [ ]:
def train_model(df):
    global FEAT_COLS
    FEAT_COLS = get_feat_cols(df)
    print(f"[MODEL] Feature Columns ({len(FEAT_COLS)}): {FEAT_COLS}")
    
    X = df[FEAT_COLS].values.astype(np.float32)
    y = df['target'].values.astype(np.float32)
    groups = df['group'].values

    gkf = GroupKFold(n_splits=CFG.N_FOLDS)
    models = []
    oof = np.zeros(len(y))
    
    for fold, (tr, va) in enumerate(gkf.split(X, y, groups)):
        print(f"  Training Fold {fold+1}/{CFG.N_FOLDS}...")
        
        # 1. Train LightGBM
        ds_tr = lgb.Dataset(X[tr], label=y[tr])
        ds_va = lgb.Dataset(X[va], label=y[va], reference=ds_tr)
        
        m_lgb = lgb.train(CFG.LGB_PARAMS, ds_tr, 
                          num_boost_round=CFG.NUM_BOOST,
                          valid_sets=[ds_va],
                          callbacks=[lgb.early_stopping(CFG.EARLY_STOP, verbose=False)])
        
        # 2. Train CatBoost (GPU-first with CPU fallback)
        try:
            m_cb = cb.CatBoostRegressor(
                iterations=1200,
                learning_rate=0.02,
                depth=4,
                l2_leaf_reg=5.0,
                loss_function='RMSE',
                random_seed=CFG.SEED,
                verbose=0,
                task_type='GPU',
                thread_count=-1
            )
            m_cb.fit(X[tr], y[tr], eval_set=(X[va], y[va]), early_stopping_rounds=50, verbose=False)
        except Exception as e:
            print(f"    CatBoost GPU failed ({e}). Retrying on CPU...")
            m_cb = cb.CatBoostRegressor(
                iterations=1200,
                learning_rate=0.02,
                depth=4,
                l2_leaf_reg=5.0,
                loss_function='RMSE',
                random_seed=CFG.SEED,
                verbose=0,
                task_type='CPU',
                thread_count=-1
            )
            m_cb.fit(X[tr], y[tr], eval_set=(X[va], y[va]), early_stopping_rounds=50, verbose=False)
        
        # 3. Train Random Forest (regularized, shallow to keep it extremely fast)
        m_rf = RandomForestRegressor(
            n_estimators=50,
            max_depth=5,
            min_samples_leaf=100,
            random_state=CFG.SEED,
            n_jobs=-1
        )
        m_rf.fit(X[tr], y[tr])
        
        models.append((m_lgb, m_cb, m_rf))
        
        # Predict validation fold
        pred_lgb = m_lgb.predict(X[va])
        pred_cb = m_cb.predict(X[va])
        pred_rf = m_rf.predict(X[va])
        
        # Weighted Ensemble Blend: 50% LightGBM + 30% CatBoost + 20% Random Forest
        oof[va] = 0.5 * pred_lgb + 0.3 * pred_cb + 0.2 * pred_rf

    df['oof_pred'] = oof
    
    # Stage-wise evaluation diagnostics
    if 'tvt_true' in df.columns:
        df['final_pred_raw'] = df['tvt_baseline'] + df['oof_pred']
        df['final_pred_smoothed'] = df['final_pred_raw'].copy()
        
        # Apply Savitzky-Golay per-well to smooth out ML prediction noise
        for g_idx, grp in df.groupby('group'):
            raw = grp['final_pred_raw'].values
            n_w = len(raw)
            if n_w > 5:
                sm = signal.savgol_filter(raw, min(CFG.SMOOTH_WIN, n_w // 2 * 2 + 1), CFG.SMOOTH_POLY)
                df.loc[grp.index, 'final_pred_smoothed'] = sm
                
        plane_rmse = np.sqrt(np.mean((df['tvt_plane'].values - df['tvt_true'].values)**2))
        raw_rmse = np.sqrt(np.mean((df['final_pred_raw'].values - df['tvt_true'].values)**2))
        sm_rmse = np.sqrt(np.mean((df['final_pred_smoothed'].values - df['tvt_true'].values)**2))
        
        print("\n" + "=" * 50)
        print("MULTIPLE-STAGE OOF RMSE EVALUATION:")
        print("=" * 50)
        print(f"Stage 1 (Regularized Dipping Plane) RMSE: {plane_rmse:.4f} ft")
        print(f"Stage 2 (Stage 1 + Ensemble Corrector) RMSE:  {raw_rmse:.4f} ft")
        print(f"Stage 3 (Stage 2 + Savitzky-Golay Smooth) RMSE: {sm_rmse:.4f} ft")
        print("=" * 50)
        
    return models


## Test Set Inference Pipeline


In [ ]:
def predict_test(models, sub_df, tree, df_slopes):
    test_wids = sub_df['well_id'].unique()
    preds = {}
    
    for wid in test_wids:
        hw, tw = load_well(wid, CFG.TEST_DIR)
        ps = find_ps_test(sub_df, wid)
        feat = build_features(hw, tw, ps, tree, df_slopes, is_train=False)
        pz = feat.iloc[ps+1:]
        X = pz[FEAT_COLS].values.astype(np.float32)
        
        # Ensemble prediction of residuals across all folds and all three models
        preds_list = []
        for m_lgb, m_cb, m_rf in models:
            p_lgb = m_lgb.predict(X)
            p_cb = m_cb.predict(X)
            p_rf = m_rf.predict(X)
            p_blend = 0.5 * p_lgb + 0.3 * p_cb + 0.2 * p_rf
            preds_list.append(p_blend)
            
        residual = np.mean(preds_list, axis=0)
        final_preds = feat['tvt_baseline'].values[ps+1:] + residual
        
        # Apply physical Savitzky-Golay smoothing to eliminate jagged steps
        final_smoothed = signal.savgol_filter(final_preds, min(CFG.SMOOTH_WIN, len(X)//2*2+1), CFG.SMOOTH_POLY)
        
        well_sub = sub_df[sub_df['well_id'] == wid].sort_values('row_idx')
        for j, (_, row) in enumerate(well_sub.iterrows()):
            preds[row['id']] = float(final_smoothed[j]) if j < len(final_smoothed) else float(final_smoothed[-1])
            
    return preds


## Main Pipeline Orchestration


In [ ]:
def main():
    t0 = time.time()
    print("=" * 60)
    print("ROGII v5: Spatial Plane prior + LGB + CatBoost + RF Ensemble")
    print("=" * 60)

    print("\n[1] Precomputing Spatial Neighbor Slopes...")
    tree, df_slopes = precompute_spatial_slopes()

    print("\n[2] Preparing Training Data (Regularized dipping plane)...")
    train_df = prepare_train(tree, df_slopes)
    
    print("\n[3] Training Residual Corrector (RMSE Objective)...")
    models = train_model(train_df)

    del train_df; gc.collect()

    print("\n[4] Generating Test Predictions...")
    sub_raw = pd.read_csv(CFG.SAMPLE_SUB)
    sub_raw['well_id'] = sub_raw['id'].apply(lambda x: x.rsplit('_', 1)[0])
    sub_raw['row_idx'] = sub_raw['id'].apply(lambda x: int(x.rsplit('_', 1)[1]))
    
    preds = predict_test(models, sub_raw, tree, df_slopes)

    out = pd.DataFrame({'id': sub_raw['id']})
    out['tvt'] = out['id'].map(preds).fillna(0.0)
    out_path = os.path.join(CFG.OUTPUT_DIR, 'submission.csv')
    out.to_csv(out_path, index=False)
    print(f"\n[DONE] {out_path} created in {(time.time()-t0)/60:.2f} min")


## Pipeline Execution

Run the end-to-end physics-informed ensemble pipeline to train the models, generate OOF evaluation metrics, predict the test wells, and write the Kaggle submission file `submission.csv`.

In [ ]:
main()